In [0]:
from datetime import datetime, timezone

from google.transit import gtfs_realtime_pb2
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Bronze contains immutable raw protobuf snapshots.
# Silver decodes them into typed, queryable GTFS-Realtime entities.

CATALOG = "bucharest_mobility"
BRONZE = "bronze"
SILVER = "silver"

SOURCE_TABLE = f"{CATALOG}.{BRONZE}.gtfs_realtime_snapshots"

In [0]:
# Helpers for decoding protobuf enums, Unix timestamps
# and multilingual GTFS-Realtime translated strings.

def enum_name(field, value):
    return field.enum_type.values_by_number[value].name


def unix_to_datetime(value):
    if not value:
        return None

    return datetime.fromtimestamp(
        value,
        tz=timezone.utc,
    ).replace(tzinfo=None)


def translation_text(translated_string, language):
    for translation in translated_string.translation:
        if translation.language == language:
            return translation.text

    return None

In [0]:
trip_update_rows = []
stop_time_update_rows = []

# Trip Updates contain a parent trip-level entity and repeated
# stop_time_update records. Silver normalizes them into two tables.

bronze_trip_snapshots = (
    spark.table(SOURCE_TABLE)
    .filter(F.col("feed_type") == "trip_updates")
    .select(
        "feed_timestamp",
        "ingested_at",
        "payload_hash",
        "payload",
    )
    .collect()
)

for snapshot in bronze_trip_snapshots:
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(bytes(snapshot["payload"]))

    for entity in feed.entity:
        if not entity.HasField("trip_update"):
            continue

        update = entity.trip_update
        trip = update.trip

        trip_update_rows.append({
            "snapshot_hash": snapshot["payload_hash"],
            "feed_timestamp": unix_to_datetime(snapshot["feed_timestamp"]),
            "ingested_at": snapshot["ingested_at"],
            "entity_id": entity.id,
            "trip_id": trip.trip_id or None,
            "route_id": trip.route_id or None,
            "start_date": trip.start_date or None,
            "start_time": trip.start_time or None,
            "trip_schedule_relationship": enum_name(
                trip.DESCRIPTOR.fields_by_name["schedule_relationship"],
                trip.schedule_relationship,
            ),
            "vehicle_id": (
                update.vehicle.id
                if update.HasField("vehicle")
                else None
            ),
            "vehicle_label": (
                update.vehicle.label
                if update.HasField("vehicle")
                else None
            ),
            "license_plate": (
                update.vehicle.license_plate
                if update.HasField("vehicle")
                else None
            ),
            "trip_update_timestamp": (
                unix_to_datetime(update.timestamp)
                if update.timestamp
                else None
            ),
        })

        for stop_update in update.stop_time_update:
            stop_time_update_rows.append({
                "snapshot_hash": snapshot["payload_hash"],
                "feed_timestamp": unix_to_datetime(snapshot["feed_timestamp"]),
                "ingested_at": snapshot["ingested_at"],
                "entity_id": entity.id,
                "trip_id": trip.trip_id or None,
                "route_id": trip.route_id or None,
                "stop_id": stop_update.stop_id or None,
                "stop_sequence": (
                    stop_update.stop_sequence
                    if stop_update.stop_sequence
                    else None
                ),
                "arrival_time": (
                    unix_to_datetime(stop_update.arrival.time)
                    if stop_update.HasField("arrival")
                    and stop_update.arrival.time
                    else None
                ),
                "arrival_delay": (
                    stop_update.arrival.delay
                    if stop_update.HasField("arrival")
                    and stop_update.arrival.HasField("delay")
                    else None
                ),
                "departure_time": (
                    unix_to_datetime(stop_update.departure.time)
                    if stop_update.HasField("departure")
                    and stop_update.departure.time
                    else None
                ),
                "departure_delay": (
                    stop_update.departure.delay
                    if stop_update.HasField("departure")
                    and stop_update.departure.HasField("delay")
                    else None
                ),
                "schedule_relationship": enum_name(
                    stop_update.DESCRIPTOR.fields_by_name[
                        "schedule_relationship"
                    ],
                    stop_update.schedule_relationship,
                ),
            })

print(f"Trip updates: {len(trip_update_rows):,}")
print(f"Stop time updates: {len(stop_time_update_rows):,}")

In [0]:
vehicle_position_rows = []

# Parse each Vehicle Position entity into one typed observation
# per realtime snapshot.

bronze_vehicle_snapshots = (
    spark.table(SOURCE_TABLE)
    .filter(F.col("feed_type") == "vehicle_positions")
    .select(
        "feed_timestamp",
        "ingested_at",
        "payload_hash",
        "payload",
    )
    .collect()
)

for snapshot in bronze_vehicle_snapshots:
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(bytes(snapshot["payload"]))

    for entity in feed.entity:
        if not entity.HasField("vehicle"):
            continue

        vehicle = entity.vehicle
        trip = vehicle.trip
        position = vehicle.position

        vehicle_position_rows.append({
            "snapshot_hash": snapshot["payload_hash"],
            "feed_timestamp": unix_to_datetime(snapshot["feed_timestamp"]),
            "ingested_at": snapshot["ingested_at"],
            "entity_id": entity.id,

            "trip_id": trip.trip_id or None,
            "route_id": trip.route_id or None,
            "start_date": trip.start_date or None,

            "trip_schedule_relationship": enum_name(
                trip.DESCRIPTOR.fields_by_name["schedule_relationship"],
                trip.schedule_relationship,
            ),

            "vehicle_id": vehicle.vehicle.id or None,
            "license_plate": vehicle.vehicle.license_plate or None,

            "latitude": float(position.latitude),
            "longitude": float(position.longitude),

            "bearing": (
                float(position.bearing)
                if position.HasField("bearing")
                else None
            ),

            "current_stop_sequence": (
                vehicle.current_stop_sequence
                if vehicle.current_stop_sequence
                else None
            ),

            "current_status": enum_name(
                vehicle.DESCRIPTOR.fields_by_name["current_status"],
                vehicle.current_status,
            ),

            "stop_id": vehicle.stop_id or None,

            "vehicle_timestamp": (
                unix_to_datetime(vehicle.timestamp)
                if vehicle.timestamp
                else None
            ),
        })

print(f"Vehicle positions: {len(vehicle_position_rows):,}")

In [0]:
service_alert_rows = []
alert_entity_rows = []

# Parse raw Service Alerts protobuf snapshots into:
#   1. one row per alert
#   2. one row per meaningful entity affected by that alert
bronze_alert_snapshots = (
    spark.table(SOURCE_TABLE)
    .filter(F.col("feed_type") == "service_alerts")
    .select(
        "feed_timestamp",
        "ingested_at",
        "payload_hash",
        "payload",
    )
    .collect()
)

for snapshot in bronze_alert_snapshots:
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(bytes(snapshot["payload"]))

    for entity in feed.entity:
        if not entity.HasField("alert"):
            continue

        alert = entity.alert

        active_start = None
        active_end = None

        # The current TPBI feed uses a single relevant active period.
        # Keep the first period in the parent alert record.
        if len(alert.active_period) > 0:
            period = alert.active_period[0]

            if period.start:
                active_start = unix_to_datetime(period.start)

            if period.end:
                active_end = unix_to_datetime(period.end)

        service_alert_rows.append({
            "snapshot_hash": snapshot["payload_hash"],
            "feed_timestamp": unix_to_datetime(
                snapshot["feed_timestamp"]
            ),
            "ingested_at": snapshot["ingested_at"],
            "alert_id": entity.id,
            "active_start": active_start,
            "active_end": active_end,
            "cause": enum_name(
                alert.DESCRIPTOR.fields_by_name["cause"],
                alert.cause,
            ),
            "effect": enum_name(
                alert.DESCRIPTOR.fields_by_name["effect"],
                alert.effect,
            ),
            "header_text_ro": translation_text(
                alert.header_text,
                "ro",
            ),
            "header_text_en": translation_text(
                alert.header_text,
                "en",
            ),
            "description_text_ro": translation_text(
                alert.description_text,
                "ro",
            ),
            "description_text_en": translation_text(
                alert.description_text,
                "en",
            ),
            "url_ro": translation_text(
                alert.url,
                "ro",
            ),
            "url_en": translation_text(
                alert.url,
                "en",
            ),
        })

        # Alerts may target multiple routes, trips, agencies or stops.
        # Store these repeated relationships in a separate child table.
        for informed in alert.informed_entity:
            agency_id = informed.agency_id or None
            route_id = informed.route_id or None

            trip_id = (
                informed.trip.trip_id
                if (
                    informed.HasField("trip")
                    and informed.trip.trip_id
                )
                else None
            )

            stop_id = informed.stop_id or None

            # TPBI occasionally emits completely empty informed_entity
            # records. They carry no analytical information and would
            # otherwise create duplicate all-null child records.
            if not any([
                agency_id,
                route_id,
                trip_id,
                stop_id,
            ]):
                continue

            alert_entity_rows.append({
                "snapshot_hash": snapshot["payload_hash"],
                "feed_timestamp": unix_to_datetime(
                    snapshot["feed_timestamp"]
                ),
                "alert_id": entity.id,
                "agency_id": agency_id,
                "route_id": route_id,
                "trip_id": trip_id,
                "stop_id": stop_id,
            })

print(f"Service alerts: {len(service_alert_rows):,}")
print(
    f"Alert informed entities: "
    f"{len(alert_entity_rows):,}"
)

In [0]:
# Explicit schemas make the Silver contract predictable and prevent
# Spark inference from changing types across ingestion runs.

trip_updates_schema = T.StructType([
    T.StructField("snapshot_hash", T.StringType(), False),
    T.StructField("feed_timestamp", T.TimestampType(), True),
    T.StructField("ingested_at", T.TimestampType(), False),
    T.StructField("entity_id", T.StringType(), False),
    T.StructField("trip_id", T.StringType(), True),
    T.StructField("route_id", T.StringType(), True),
    T.StructField("start_date", T.StringType(), True),
    T.StructField("start_time", T.StringType(), True),
    T.StructField("trip_schedule_relationship", T.StringType(), True),
    T.StructField("vehicle_id", T.StringType(), True),
    T.StructField("vehicle_label", T.StringType(), True),
    T.StructField("license_plate", T.StringType(), True),
    T.StructField("trip_update_timestamp", T.TimestampType(), True),
])

stop_time_updates_schema = T.StructType([
    T.StructField("snapshot_hash", T.StringType(), False),
    T.StructField("feed_timestamp", T.TimestampType(), True),
    T.StructField("ingested_at", T.TimestampType(), False),
    T.StructField("entity_id", T.StringType(), False),
    T.StructField("trip_id", T.StringType(), True),
    T.StructField("route_id", T.StringType(), True),
    T.StructField("stop_id", T.StringType(), True),
    T.StructField("stop_sequence", T.IntegerType(), True),
    T.StructField("arrival_time", T.TimestampType(), True),
    T.StructField("arrival_delay", T.IntegerType(), True),
    T.StructField("departure_time", T.TimestampType(), True),
    T.StructField("departure_delay", T.IntegerType(), True),
    T.StructField("schedule_relationship", T.StringType(), True),
])

vehicle_positions_schema = T.StructType([
    T.StructField("snapshot_hash", T.StringType(), False),
    T.StructField("feed_timestamp", T.TimestampType(), True),
    T.StructField("ingested_at", T.TimestampType(), False),
    T.StructField("entity_id", T.StringType(), False),
    T.StructField("trip_id", T.StringType(), True),
    T.StructField("route_id", T.StringType(), True),
    T.StructField("start_date", T.StringType(), True),
    T.StructField("trip_schedule_relationship", T.StringType(), True),
    T.StructField("vehicle_id", T.StringType(), True),
    T.StructField("license_plate", T.StringType(), True),
    T.StructField("latitude", T.DoubleType(), True),
    T.StructField("longitude", T.DoubleType(), True),
    T.StructField("bearing", T.DoubleType(), True),
    T.StructField("current_stop_sequence", T.IntegerType(), True),
    T.StructField("current_status", T.StringType(), True),
    T.StructField("stop_id", T.StringType(), True),
    T.StructField("vehicle_timestamp", T.TimestampType(), True),
])

service_alerts_schema = T.StructType([
    T.StructField("snapshot_hash", T.StringType(), False),
    T.StructField("feed_timestamp", T.TimestampType(), True),
    T.StructField("ingested_at", T.TimestampType(), False),
    T.StructField("alert_id", T.StringType(), False),
    T.StructField("active_start", T.TimestampType(), True),
    T.StructField("active_end", T.TimestampType(), True),
    T.StructField("cause", T.StringType(), True),
    T.StructField("effect", T.StringType(), True),
    T.StructField("header_text_ro", T.StringType(), True),
    T.StructField("header_text_en", T.StringType(), True),
    T.StructField("description_text_ro", T.StringType(), True),
    T.StructField("description_text_en", T.StringType(), True),
    T.StructField("url_ro", T.StringType(), True),
    T.StructField("url_en", T.StringType(), True),
])

alert_entities_schema = T.StructType([
    T.StructField("snapshot_hash", T.StringType(), False),
    T.StructField("feed_timestamp", T.TimestampType(), True),
    T.StructField("alert_id", T.StringType(), False),
    T.StructField("agency_id", T.StringType(), True),
    T.StructField("route_id", T.StringType(), True),
    T.StructField("trip_id", T.StringType(), True),
    T.StructField("stop_id", T.StringType(), True),
])

In [0]:
# Convert the decoded Python records into typed Spark DataFrames
# ready for Delta persistence.

trip_updates_df = spark.createDataFrame(
    trip_update_rows,
    trip_updates_schema,
)

stop_time_updates_df = spark.createDataFrame(
    stop_time_update_rows,
    stop_time_updates_schema,
)

vehicle_positions_df = spark.createDataFrame(
    vehicle_position_rows,
    vehicle_positions_schema,
)

service_alerts_df = spark.createDataFrame(
    service_alert_rows,
    service_alerts_schema,
)

alert_entities_df = spark.createDataFrame(
    alert_entity_rows,
    alert_entities_schema,
)

print("Silver DataFrames created.")

In [0]:
# Silver is append-only by source snapshot. Existing snapshot hashes
# are excluded so rerunning the transformation remains idempotent.

def append_new_snapshots(df, table_name):
    target_table = f"{CATALOG}.{SILVER}.{table_name}"

    if spark.catalog.tableExists(target_table):
        existing_hashes = (
            spark.table(target_table)
            .select("snapshot_hash")
            .distinct()
        )

        df = df.join(
            existing_hashes,
            on="snapshot_hash",
            how="left_anti",
        )

    new_rows = df.count()

    if new_rows == 0:
        print(f"SKIP {table_name}: no new snapshots")
        return

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table)
    )

    print(f"LOADED {table_name}: {new_rows:,} rows")

In [0]:
# Persist newly decoded realtime snapshots into their Silver tables.
append_new_snapshots(
    trip_updates_df,
    "trip_updates",
)

append_new_snapshots(
    stop_time_updates_df,
    "stop_time_updates",
)

append_new_snapshots(
    vehicle_positions_df,
    "vehicle_positions",
)

append_new_snapshots(
    service_alerts_df,
    "service_alerts",
)

append_new_snapshots(
    alert_entities_df,
    "alert_informed_entities",
)

In [0]:
# Operational row-count validation for the five realtime Silver tables.
SILVER_REALTIME_TABLES = [
    "trip_updates",
    "stop_time_updates",
    "vehicle_positions",
    "service_alerts",
    "alert_informed_entities",
]

for table_name in SILVER_REALTIME_TABLES:
    count = spark.table(
        f"{CATALOG}.{SILVER}.{table_name}"
    ).count()

    print(f"{table_name}: {count:,} rows")

In [0]:
# Validate the declared grain of every Silver realtime entity.
# Duplicate grains indicate a parsing or source-modeling issue.
GRAINS = {
    "trip_updates": [
        "snapshot_hash",
        "entity_id",
    ],
    "stop_time_updates": [
        "snapshot_hash",
        "entity_id",
        "stop_sequence",
    ],
    "vehicle_positions": [
        "snapshot_hash",
        "entity_id",
    ],
    "service_alerts": [
        "snapshot_hash",
        "alert_id",
    ],
    "alert_informed_entities": [
        "snapshot_hash",
        "alert_id",
        "agency_id",
        "route_id",
        "trip_id",
        "stop_id",
    ],
}

for table_name, keys in GRAINS.items():
    df = spark.table(
        f"{CATALOG}.{SILVER}.{table_name}"
    )

    duplicates = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicates == 0, (
        f"{table_name}: {duplicates} duplicate grains"
    )

    print(f"✓ {table_name}: grain unique")

In [0]:
# Measure referential coverage between realtime identifiers and
# the available GTFS Static Silver data. Partial matches are expected
# because static and realtime snapshots are captured independently.
checks = [
    (
        "trip_updates",
        "trip_id",
        "trips",
        "trip_id",
    ),
    (
        "stop_time_updates",
        "trip_id",
        "trips",
        "trip_id",
    ),
    (
        "stop_time_updates",
        "stop_id",
        "stops",
        "stop_id",
    ),
    (
        "vehicle_positions",
        "trip_id",
        "trips",
        "trip_id",
    ),
    (
        "vehicle_positions",
        "stop_id",
        "stops",
        "stop_id",
    ),
]

for realtime_table, realtime_key, static_table, static_key in checks:

    realtime = (
        spark.table(
            f"{CATALOG}.{SILVER}.{realtime_table}"
        )
        .select(realtime_key)
        .filter(F.col(realtime_key).isNotNull())
        .distinct()
    )

    static = (
        spark.table(
            f"{CATALOG}.{SILVER}.{static_table}"
        )
        .select(static_key)
        .distinct()
    )

    unmatched = (
        realtime.join(
            static,
            realtime[realtime_key] == static[static_key],
            "left_anti",
        )
        .count()
    )

    total = realtime.count()

    matched = total - unmatched

    match_pct = (
        matched / total * 100
        if total > 0
        else 0
    )

    print(
        f"{realtime_table}.{realtime_key} → "
        f"{static_table}.{static_key}: "
        f"{matched:,}/{total:,} matched "
        f"({match_pct:.2f}%)"
    )